# Machine Learning Lab: Classifying Behavioral Function from Functional Analysis Data

In this lab, you will use supervised machine learning to classify the function of problem behavior from functional analysis (FA) summary data. This mirrors a task clinicians perform routinely: examining response rates across FA conditions to determine whether behavior is maintained by attention, escape, tangible reinforcement, or automatic reinforcement.

You will:
1. Explore the dataset and understand its structure
2. Build a decision tree classifier
3. Evaluate model performance with a confusion matrix and classification report
4. Investigate overfitting across different tree depths
5. Build a random forest and compare to the single tree
6. Discuss the prediction-explanation tradeoff

**Objectives:**
- Gain hands-on experience with scikit-learn's classification tools
- Understand how decision trees make classification decisions
- Appreciate the tradeoff between model complexity and interpretability

## Setup

Run the cell below to import the libraries you will need.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report,
)

np.random.seed(42)

## Task 1: Load and Explore the Data

Load the file `fa_classification_data.csv`. This dataset contains simulated functional analysis summary data for 60 participants. Each participant has:

- `attention_rate`: response rate during the attention condition
- `escape_rate`: response rate during the escape condition
- `tangible_rate`: response rate during the tangible condition
- `play_rate`: response rate during the play (control) condition
- `function`: the true behavioral function (attention, escape, tangible, or automatic)

Tasks:
1. Load the CSV file into a pandas DataFrame
2. Print the shape and first few rows
3. Check the distribution of the `function` column
4. Compute descriptive statistics for each rate column grouped by function

In [ ]:
df = pd.read_csv("fa_classification_data.csv")
print("Shape:", df.shape)
print(df.head())

In [ ]:
feature_cols = ['attention_rate', 'escape_rate', 'tangible_rate', 'play_rate']

print("Class distribution:")
print(df['function'].value_counts())
print("\nMean rate by function:")
print(df.groupby('function')[feature_cols].mean())

## Task 2: Prepare Features and Labels

Separate the data into:
- **X** (features): the four rate columns
- **y** (labels): the `function` column

Then split into training (75%) and test (25%) sets using `train_test_split` with `random_state=42` and `stratify=y` to ensure each function is represented proportionally in both sets.

Print the size of each set and verify the class distribution in both.

In [ ]:
X = df[feature_cols]
y = df['function']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y)

print("Train set:", X_train.shape, " Test set:", X_test.shape)
print("\nTrain class counts:\n", y_train.value_counts())
print("\nTest class counts:\n", y_test.value_counts())

## Task 3: Train a Decision Tree Classifier

Train a `DecisionTreeClassifier` with `random_state=42` (use default hyperparameters for now).

After training:
1. Print the training accuracy
2. Print the test accuracy
3. Note whether there is a gap between the two (this is your first signal about overfitting)

In [ ]:
tree = DecisionTreeClassifier(random_state=42)
tree.fit(X_train, y_train)

train_acc = accuracy_score(y_train, tree.predict(X_train))
test_acc = accuracy_score(y_test, tree.predict(X_test))
print(f"Training accuracy: {train_acc:.3f}")
print(f"Test accuracy:     {test_acc:.3f}")
print(f"Gap (train - test): {train_acc - test_acc:.3f}")

## Task 4: Visualize the Decision Tree

Use `sklearn.tree.plot_tree` to visualize the trained decision tree.

Tips:
- Use `filled=True` for color-coded nodes
- Use `feature_names` and `class_names` for readability
- Set `figsize` large enough to read the tree (e.g., 20x10)

Examine the tree: What features does it split on first? Does the tree structure make intuitive sense given how FA data are typically interpreted?

In [ ]:
fig, ax = plt.subplots(figsize=(20, 10))
plot_tree(tree, filled=True, feature_names=list(X.columns),
          class_names=list(tree.classes_), fontsize=9, ax=ax)
plt.title("Default Decision Tree")
plt.show()

*Describe what you observe about the tree structure here.*

## Task 5: Confusion Matrix and Classification Report

Evaluate the decision tree on the **test set**:

1. Compute and display the confusion matrix using `ConfusionMatrixDisplay`
2. Print the full `classification_report` (precision, recall, F1 for each class)

Which function is easiest to classify? Which is hardest? Why might that be?

In [ ]:
y_pred = tree.predict(X_test)

fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, ax=ax)
ax.set_title("Decision Tree - Test Set")
plt.show()

print(classification_report(y_test, y_pred))

*Interpret the confusion matrix and classification report here.*

## Task 6: Investigate Overfitting Across Tree Depths

Decision trees are prone to overfitting -- they can memorize the training data perfectly but perform poorly on new data. The `max_depth` parameter controls the tree's complexity.

1. Train decision trees with `max_depth` values from 1 to 10
2. For each depth, record the training accuracy and test accuracy
3. Plot both curves on the same figure

Identify the `max_depth` value that gives the best test accuracy. What happens to the gap between training and test accuracy as depth increases?

In [ ]:
depths = list(range(1, 11))
train_scores, test_scores = [], []
for d in depths:
    t = DecisionTreeClassifier(max_depth=d, random_state=42)
    t.fit(X_train, y_train)
    train_scores.append(accuracy_score(y_train, t.predict(X_train)))
    test_scores.append(accuracy_score(y_test, t.predict(X_test)))

best_depth = depths[int(np.argmax(test_scores))]

plt.figure(figsize=(8, 5))
plt.plot(depths, train_scores, 'o-', label='Training accuracy')
plt.plot(depths, test_scores, 's-', label='Test accuracy')
plt.axvline(best_depth, color='gray', linestyle='--', alpha=0.7,
            label=f'Best depth = {best_depth}')
plt.xlabel('max_depth'); plt.ylabel('Accuracy')
plt.title('Overfitting vs. Tree Depth'); plt.legend()
plt.show()
print("Best max_depth by test accuracy:", best_depth,
      f"(test acc = {max(test_scores):.3f})")

*Describe the overfitting pattern and the optimal max_depth here.*

## Task 7: Build a Random Forest

A random forest is an ensemble of many decision trees, each trained on a random subset of the data and features. The ensemble averages over individual trees' errors, typically producing better generalization.

1. Train a `RandomForestClassifier` with `n_estimators=100` and `random_state=42`
2. Report training and test accuracy
3. Compare to the best single decision tree from Task 6
4. Print the feature importances -- which FA condition rates are most important for classification?

In [ ]:
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

rf_train = accuracy_score(y_train, rf.predict(X_train))
rf_test = accuracy_score(y_test, rf.predict(X_test))
print(f"Random Forest training accuracy: {rf_train:.3f}")
print(f"Random Forest test accuracy:     {rf_test:.3f}")
print(f"(Best single tree test accuracy was {max(test_scores):.3f})")

print("\nFeature importances:")
for name, imp in zip(X.columns, rf.feature_importances_):
    print(f"  {name}: {imp:.3f}")

## Task 8: Visualize Feature Importances

Create a bar chart of the random forest's feature importances. Label the bars with the feature names.

Do the importances align with your intuition about how clinicians interpret FA data? Are all four condition rates equally useful, or do some carry more information?

In [ ]:
importances = rf.feature_importances_
order = np.argsort(importances)[::-1]

plt.figure(figsize=(8, 5))
plt.bar([X.columns[i] for i in order], importances[order], color='steelblue')
plt.ylabel('Importance')
plt.title('Random Forest Feature Importances')
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

*Interpret the feature importances here.*

## Task 9: The Prediction-Explanation Tradeoff

Summarize your results in a comparison table:

| Model | Training Accuracy | Test Accuracy | Interpretable? |
|-------|-------------------|---------------|----------------|
| Decision Tree (default) | ? | ? | ? |
| Decision Tree (best depth) | ? | ? | ? |
| Random Forest | ? | ? | ? |

Then write a discussion (3-4 paragraphs) addressing:

1. **Prediction vs. explanation**: The random forest likely predicts better, but can you explain *why* it classifies a given case the way it does? How does this compare to the single decision tree? In clinical behavior analysis, is prediction or explanation more important?

2. **Clinical utility**: Could a classifier like this be useful in practice? What are the limitations of training on simulated data? What real-world complications would arise (e.g., undifferentiated functions, multiply-maintained behavior, variable FA protocols)?

3. **Modeling philosophy**: How does the ML approach here differ from the parametric models (e.g., matching law, demand curves) we have used earlier in the course? What do we gain and lose by moving from theory-driven to data-driven modeling?

*Write your discussion here.*